# 03 · Phase 4: SeqFT vs CIVD (one continual-learning stage)

**Stage 0 (already measured, R1b):** `lerobot/smolvla_libero`, trained on all 10 LIBERO-Goal tasks.
**Stage 1 (this notebook):** fine-tune it further on **one** Goal task (`NEW_TASK`, its weakest task in R1b), then
re-evaluate the new task and old tasks to measure forgetting.

- **GPU0 · M0 SeqFT:** plain fine-tune on the new task.
- **GPU1 · M3 CIVD:** same, plus velocity distillation from the frozen stage-0 model on the *same* new-task
  observations paired with the *other* tasks' instructions (`code/civd/civd.py`). No old demonstrations are used.

Both runs use identical steps, batch, lr, seed. Training is a short burst (T4 has no native bf16, ~9 s/step at batch 32),
so the question is whether a short single-task fine-tune already damages old tasks, and whether CIVD reduces it.

Run with **Save Version → Save & Run All (Commit)**, GPU T4 x2, Internet On.
Outputs in `/kaggle/working/cl/`: `summary.md`, `progress.log`, `train_m0.log`, `train_m3.log`, `eval/*/eval_info.json`.

In [ ]:
# ---- parameters ----
SUITE, BASE = "libero_goal", "lerobot/smolvla_libero"
NEW_TASK = 3                   # weakest Goal task in R1b (60 %): open the top drawer and put the bowl inside
OLD_TASKS = [8, 4, 0, 1, 5, 7, 9, 2, 6]   # all old Goal tasks; bowl tasks (8, 4) first
STEPS, BATCH = 1200, 16
CIVD_LAMBDA, CIVD_DISTILL_BS = 1.0, 4
TRAIN_SEED = 1000              # training seed; evaluation always uses seed 1000 (same initial states as stage 0)
# one run per GPU: (name, mode, lambda); mode = seqft | civd (old instructions) | civdnew (M3b: new instruction)
RUNS = [("m0", "seqft", 0.0), ("m3", "civd", CIVD_LAMBDA)]
EVAL_EPISODES, NAS = 10, 10    # same as R1b
SMOKE_STEPS = 3                # short CIVD run first: distill loss must be ~0 at step 1 (student == teacher)
open("/tmp/params.sh", "w").write(
    f'SUITE={SUITE}\nBASE={BASE}\nNEW_TASK={NEW_TASK}\nOLD_TASKS="{" ".join(map(str, OLD_TASKS))}"\n'
    f'STEPS={STEPS}\nBATCH={BATCH}\nCIVD_LAMBDA={CIVD_LAMBDA}\nCIVD_DISTILL_BS={CIVD_DISTILL_BS}\n'
    f'EVAL_EPISODES={EVAL_EPISODES}\nNAS={NAS}\nSMOKE_STEPS={SMOKE_STEPS}\nTRAIN_SEED={TRAIN_SEED}\n'
    + "".join(f'RUN{g}="{n} {m} {l}"\n' for g, (n, m, l) in enumerate(RUNS)))
print(open("/tmp/params.sh").read())

In [ ]:
%%bash
echo "=== ENV ===" >> /kaggle/working/cl_report.txt
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv | tee -a /kaggle/working/cl_report.txt
echo "CPUs: $(nproc)" | tee -a /kaggle/working/cl_report.txt
free -g | head -2 | tee -a /kaggle/working/cl_report.txt

In [ ]:
%%bash
# System libraries for headless MuJoCo rendering and video decoding
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq libegl1 libgl1 libglvnd0 libosmesa6 libosmesa6-dev ffmpeg cmake build-essential > /dev/null 2>&1 \
  && echo "apt: ok" | tee -a /kaggle/working/cl_report.txt || { echo "apt: FAILED" | tee -a /kaggle/working/cl_report.txt; exit 1; }
# LeRobot needs Python >= 3.12; build an isolated venv with uv
pip install -q -U uv
uv python install 3.12 > /dev/null 2>&1
rm -rf /tmp/venv && uv venv /tmp/venv --python 3.12 > /dev/null 2>&1
/tmp/venv/bin/python --version > /dev/null 2>&1 \
  && echo "venv: ok ($(/tmp/venv/bin/python --version))" | tee -a /kaggle/working/cl_report.txt || { echo "venv: FAILED" | tee -a /kaggle/working/cl_report.txt; exit 1; }

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
LOG=/kaggle/working/install.log
cd /tmp && rm -rf lerobot && git clone -q https://github.com/huggingface/lerobot.git && cd lerobot && git checkout -q ba83e4cf43737e2665be0aafcb5f513b83c7d0f3
echo "lerobot commit: $(git rev-parse HEAD)" | tee -a /kaggle/working/cl_report.txt
unset UV_PYTHON UV_PROJECT_ENVIRONMENT PIP_TARGET
# Target the venv explicitly: Kaggle's environment otherwise sends uv to the system Python
uv pip install --python /tmp/venv/bin/python -e ".[smolvla,libero]" > $LOG 2>&1
RC=$?
echo "install exit: $RC ($(head -1 $LOG))" | tee -a /kaggle/working/cl_report.txt
if [ $RC -ne 0 ]; then tail -40 $LOG | tee -a /kaggle/working/cl_report.txt; exit 1; fi
python - <<'EOF' 2>&1 | tee -a /kaggle/working/cl_report.txt
import importlib, torch
for m in ["torch", "lerobot", "transformers", "mujoco", "libero"]:
    mod = importlib.import_module(m)
    print(f"import {m}: ok {getattr(mod, '__version__', '')}")
print("cuda:", torch.version.cuda, "|", torch.cuda.device_count(), "x", torch.cuda.get_device_name(0))
EOF
[ ${PIPESTATUS[0]} -eq 0 ] || exit 1
# LIBERO asks for a dataset path on first import; answer "N" to keep defaults
(yes N | python -c "from libero.libero import benchmark; print('libero:', len(benchmark.get_benchmark_dict()), 'suites')") 2>&1 | tail -1 | tee -a /kaggle/working/cl_report.txt
# Kaggle ships its own /usr/local/bin/lerobot-* that shadow the venv's; every call below uses `/tmp/venv/bin/python -m ...`
echo "lerobot-eval on PATH: $(which -a lerobot-eval | tr '\n' ' ')" | tee -a /kaggle/working/cl_report.txt
/tmp/venv/bin/python -c "import lerobot, libero, mujoco" && touch /tmp/setup_ok && echo "setup: ok" | tee -a /kaggle/working/cl_report.txt

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
mkdir -p /kaggle/working/cl/eval /tmp/cl
cat > /tmp/cl/civd.py <<'CIVD_EOF'
"""CIVD: Cross-Instruction Velocity Distillation for SmolVLA (LeRobot ba83e4c).

Stage-k training loss:
    L = L_FM(new-task batch) + lambda_d * E || v_student(x_t, t, o, l_old) - v_teacher(x_t, t, o, l_old) ||^2

- o: observations (images + state) of the CURRENT new-task batch, no old demos are stored.
- l_old: instructions of previously learned tasks (strings only).
- teacher: frozen copy of the previous-stage checkpoint.
- x_t = t * eps + (1 - t) * A_T, with A_T sampled by the teacher under l_old (NOISE_AT_ONE convention,
  same as VLAFlowMatching.forward), so the student is supervised on the old task's flow, not the new one.
- mode "new" (ablation M3b): distill with the batch's own (new) instruction instead of l_old.

Usage: this file is a drop-in launcher for lerobot_train. All CLI args are passed through to
LeRobot; CIVD is configured with environment variables:
    CIVD_TEACHER      path to the previous-stage `pretrained_model` dir (required)
    CIVD_OLD_TASKS    old instructions separated by "|" (required for mode "old")
    CIVD_LAMBDA       distillation weight (default 1.0)
    CIVD_DISTILL_BS   how many batch samples get the distillation term (default 8)
    CIVD_MODE         "old" (CIVD, default) or "new" (M3b ablation)
    CIVD_LOG_EVERY    print distill stats every N steps (default 100)
"""

import os
import random
import sys
import time

import torch
import torch.nn.functional as F

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy, make_att_2d_masks
from lerobot.utils.constants import OBS_LANGUAGE_ATTENTION_MASK, OBS_LANGUAGE_TOKENS


def velocity(model, images, img_masks, lang_tokens, lang_masks, state, x_t, time):
    """v_theta(x_t, t | o, l). Same computation as VLAFlowMatching.forward, without the loss."""
    prefix_embs, prefix_pad_masks, prefix_att_masks = model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state
    )
    suffix_embs, suffix_pad_masks, suffix_att_masks = model.embed_suffix(x_t, time)
    pad_masks = torch.cat([prefix_pad_masks, suffix_pad_masks], dim=1)
    att_masks = torch.cat([prefix_att_masks, suffix_att_masks], dim=1)
    att_2d_masks = make_att_2d_masks(pad_masks, att_masks)
    position_ids = torch.cumsum(pad_masks, dim=1) - 1
    (_, suffix_out), _ = model.vlm_with_expert.forward(
        attention_mask=att_2d_masks,
        position_ids=position_ids,
        past_key_values=None,
        inputs_embeds=[prefix_embs, suffix_embs],
        use_cache=False,
    )
    suffix_out = suffix_out[:, -model.config.chunk_size :].to(dtype=torch.float32)
    return model.action_out_proj(suffix_out)


class CIVD:
    def __init__(self, student: SmolVLAPolicy):
        cfg = os.environ
        self.mode = cfg.get("CIVD_MODE", "old")
        self.lam = float(cfg.get("CIVD_LAMBDA", "1.0"))
        self.distill_bs = int(cfg.get("CIVD_DISTILL_BS", "8"))
        self.log_every = int(cfg.get("CIVD_LOG_EVERY", "100"))
        self.device = next(student.parameters()).device

        self.teacher = SmolVLAPolicy.from_pretrained(cfg["CIVD_TEACHER"]).to(self.device).eval()
        self.teacher.requires_grad_(False)

        self.old_tokens = []
        if self.mode == "old":
            from transformers import AutoTokenizer

            tok = AutoTokenizer.from_pretrained(student.config.vlm_model_name)
            for task in cfg["CIVD_OLD_TASKS"].split("|"):
                # Same text processing as the preprocessor: NewLineTaskProcessorStep + tokenizer.
                enc = tok(task.strip() + "\n", return_tensors="pt", truncation=True,
                          max_length=student.config.tokenizer_max_length)
                self.old_tokens.append(
                    (enc["input_ids"].to(self.device), enc["attention_mask"].to(self.device).bool())
                )
        self.step = 0
        self.t_extra = 0.0
        print(f"[civd] mode={self.mode} lambda={self.lam} distill_bs={self.distill_bs} "
              f"old_tasks={len(self.old_tokens)} teacher={cfg['CIVD_TEACHER']}", flush=True)

    def loss(self, student: SmolVLAPolicy, batch):
        t0 = time.perf_counter()
        n = min(self.distill_bs, batch[OBS_LANGUAGE_TOKENS].shape[0])
        sub = {k: (v[:n] if torch.is_tensor(v) else v) for k, v in batch.items()}
        images, img_masks = student.prepare_images(sub)
        state = student.prepare_state(sub)

        if self.mode == "old":
            ids, mask = random.choice(self.old_tokens)
            lang_tokens, lang_masks = ids.expand(n, -1), mask.expand(n, -1)
        else:
            lang_tokens, lang_masks = sub[OBS_LANGUAGE_TOKENS], sub[OBS_LANGUAGE_ATTENTION_MASK]

        tm = self.teacher.model
        with torch.no_grad():
            a_teacher = tm.sample_actions(images, img_masks, lang_tokens, lang_masks, state)
            eps = tm.sample_noise(a_teacher.shape, a_teacher.device)
            t = tm.sample_time(n, a_teacher.device)
            x_t = t[:, None, None] * eps + (1 - t[:, None, None]) * a_teacher
            v_teacher = velocity(tm, images, img_masks, lang_tokens, lang_masks, state, x_t, t)

        v_student = velocity(student.model, images, img_masks, lang_tokens, lang_masks, state, x_t, t)
        d = student.config.action_feature.shape[0]  # ignore padded action dims
        l_d = F.mse_loss(v_student[..., :d], v_teacher[..., :d])

        self.step += 1
        self.t_extra += time.perf_counter() - t0
        if self.step % self.log_every == 0:
            mem = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0
            print(f"[civd] step:{self.step} distill:{l_d.item():.4f} "
                  f"extra_s:{self.t_extra / self.step:.3f} peak_mem_gb:{mem:.2f}", flush=True)
        return l_d


_orig_forward = SmolVLAPolicy.forward


def _civd_forward(self, batch, noise=None, time=None, reduction="mean"):
    loss, out = _orig_forward(self, batch, noise=noise, time=time, reduction=reduction)
    if not self.training:
        return loss, out
    if not hasattr(self, "_civd"):
        self._civd = CIVD(self)
    l_d = self._civd.loss(self, batch)
    out["civd_distill"] = l_d.item()
    return loss + self._civd.lam * l_d, out


def install():
    SmolVLAPolicy.forward = _civd_forward


if __name__ == "__main__":
    install()
    from lerobot.scripts import lerobot_train

    sys.argv[0] = "lerobot_train"
    lerobot_train.main()
CIVD_EOF
cat > /tmp/libero_goal_episodes.json <<'EOF'
{"revision": "a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4", "tasks": {"0": {"instruction": "open the middle drawer of the cabinet", "task_index": 19, "episodes": [399, 405, 410, 421, 437, 443, 463, 469, 473, 490, 496, 497, 516, 519, 525, 541, 547, 552, 561, 566, 577, 585, 599, 611, 631, 633, 659, 664, 672, 678, 687, 693, 703, 722, 725, 734, 754, 765, 772, 779, 780, 792, 802]}, "1": {"instruction": "put the bowl on the stove", "task_index": 17, "episodes": [392, 394, 395, 401, 415, 424, 445, 455, 495, 500, 524, 530, 544, 545, 550, 557, 558, 565, 580, 598, 604, 612, 616, 618, 629, 637, 641, 643, 649, 673, 676, 688, 691, 692, 704, 707, 721, 740, 741, 747, 751, 763, 764, 786, 789, 796, 804, 805]}, "2": {"instruction": "put the wine bottle on top of the cabinet", "task_index": 14, "episodes": [385, 389, 396, 397, 404, 417, 419, 423, 427, 430, 432, 436, 454, 468, 470, 475, 476, 479, 480, 482, 512, 515, 528, 533, 543, 546, 554, 555, 556, 574, 588, 591, 592, 596, 619, 635, 645, 648, 656, 668, 682, 686, 697, 735, 766, 788, 795]}, "3": {"instruction": "open the top drawer and put the bowl inside", "task_index": 12, "episodes": [382, 383, 387, 409, 414, 416, 449, 462, 465, 489, 494, 498, 523, 526, 534, 538, 548, 559, 560, 576, 594, 614, 624, 627, 684, 694, 701, 711, 737, 738, 742, 758, 762, 782, 784, 790]}, "4": {"instruction": "put the bowl on top of the cabinet", "task_index": 18, "episodes": [398, 402, 438, 444, 446, 452, 456, 458, 474, 493, 501, 505, 514, 521, 570, 573, 590, 597, 601, 608, 630, 636, 638, 660, 661, 662, 665, 669, 674, 681, 689, 696, 699, 702, 706, 712, 724, 731, 732, 743, 745, 767, 775, 791, 793, 794]}, "5": {"instruction": "push the plate to the front of the stove", "task_index": 15, "episodes": [386, 400, 411, 420, 425, 442, 461, 471, 472, 478, 506, 535, 553, 584, 589, 593, 603, 606, 647, 663, 680, 685, 710, 713, 715, 733, 744, 752, 755, 761, 769, 778, 781]}, "6": {"instruction": "put the cream cheese in the bowl", "task_index": 13, "episodes": [384, 391, 393, 413, 434, 439, 457, 464, 491, 492, 508, 510, 518, 527, 536, 540, 562, 564, 581, 587, 600, 610, 632, 644, 650, 657, 677, 698, 709, 714, 720, 730, 748, 756, 760, 771, 776, 777, 787, 797]}, "7": {"instruction": "turn on the stove", "task_index": 16, "episodes": [388, 390, 406, 412, 418, 435, 440, 441, 450, 453, 460, 477, 486, 487, 499, 504, 520, 529, 531, 567, 569, 572, 575, 578, 602, 605, 609, 613, 617, 622, 628, 640, 654, 658, 690, 700, 705, 717, 723, 728, 729, 736, 746, 753, 759, 774, 783, 785, 799, 800]}, "8": {"instruction": "put the bowl on the plate", "task_index": 10, "episodes": [379, 422, 426, 431, 433, 447, 448, 451, 459, 466, 481, 483, 488, 507, 511, 513, 522, 532, 537, 549, 551, 563, 568, 582, 607, 615, 620, 621, 626, 634, 639, 642, 646, 653, 655, 670, 679, 708, 716, 718, 726, 727, 749, 750, 768, 770, 801, 803, 806]}, "9": {"instruction": "put the wine bottle on the rack", "task_index": 11, "episodes": [380, 381, 403, 407, 408, 428, 429, 467, 484, 485, 502, 503, 509, 517, 539, 542, 571, 579, 583, 586, 595, 623, 625, 651, 652, 666, 667, 671, 675, 683, 695, 719, 739, 757, 773, 798]}}}
EOF
cat > /tmp/select.py <<'EOF'
import json
from libero.libero import benchmark

p = {k: v.strip('"') for k, v in (l.strip().split("=", 1) for l in open("/tmp/params.sh") if "=" in l)}
m = json.load(open("/tmp/libero_goal_episodes.json"))
suite = benchmark.get_benchmark_dict()[p["SUITE"]]()
for t, e in m["tasks"].items():
    assert suite.get_task(int(t)).language.strip().lower() == e["instruction"], t
new = m["tasks"][p["NEW_TASK"]]
old = [e["instruction"] for t, e in m["tasks"].items() if t != p["NEW_TASK"]]
print(f"new task: {new['instruction']} | {len(new['episodes'])} episodes | revision {m['revision']}")
print(f"distillation instructions: {len(old)} other Goal tasks")
json.dump(new["episodes"], open("/tmp/cl_episodes.json", "w"))
# experience replay variants: new-task episodes + the first K episodes of every old Goal task
for k in (1, 5):
    er = sorted(new["episodes"] + [ep for t, e in m["tasks"].items() if t != p["NEW_TASK"] for ep in e["episodes"][:k]])
    json.dump(er, open(f"/tmp/cl_episodes_er{k}.json", "w"))
    print(f"ER-{k}: {len(er)} episodes ({len(er) - len(new['episodes'])} replayed from old tasks)")
open("/tmp/ds_rev", "w").write(m["revision"])
open("/tmp/old_tasks", "w").write("|".join(old))
EOF
python /tmp/select.py 2>&1 | tail -5 | tee -a /kaggle/working/cl_report.txt
[ -s /tmp/cl_episodes.json ] || { echo "no episodes selected" | tee -a /kaggle/working/cl_report.txt; exit 1; }
/tmp/venv/bin/python -m py_compile /tmp/cl/civd.py && echo "civd.py: ok" | tee -a /kaggle/working/cl_report.txt

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
source /tmp/params.sh
# Train one method on one GPU: train_one GPU NAME STEPS [civd]
cat > /tmp/train_one.sh <<'EOF'
#!/usr/bin/env bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
source /tmp/params.sh
GPU=$1; NAME=$2; N=$3; MODE=$4; LAM=${5:-$CIVD_LAMBDA}
# erK = SeqFT with K replayed episodes per old task; civderK = CIVD plus the same replay
EPF=/tmp/cl_episodes.json
case "$MODE" in
  er*) EPF=/tmp/cl_episodes_${MODE}.json; MODE=seqft ;;
  civder*) EPF=/tmp/cl_episodes_er${MODE#civder}.json; MODE=civd ;;
esac
EPS=$(tr -d ' ' < $EPF)
if [ "$MODE" = "civd" ] || [ "$MODE" = "civdnew" ]; then
  ENTRY="/tmp/cl/civd.py"
  CM=old; [ "$MODE" = "civdnew" ] && CM=new
  export CIVD_TEACHER=$BASE CIVD_OLD_TASKS="$(cat /tmp/old_tasks)" CIVD_LAMBDA=$LAM CIVD_DISTILL_BS=$CIVD_DISTILL_BS CIVD_MODE=$CM
  [ "$N" -le 10 ] && export CIVD_LOG_EVERY=1 || export CIVD_LOG_EVERY=25
else
  ENTRY="-m lerobot.scripts.lerobot_train"
fi
nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -i $GPU -lms 2000 > /tmp/mem_$NAME.txt &
MON=$!
S=$(date +%s)
CUDA_VISIBLE_DEVICES=$GPU /tmp/venv/bin/python $ENTRY \
  --policy.path=$BASE --policy.device=cuda --policy.push_to_hub=false --policy.train_expert_only=true \
  --dataset.repo_id=lerobot/libero --dataset.revision=$(cat /tmp/ds_rev) "--dataset.episodes=$EPS" \
  '--rename_map={"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}' \
  --batch_size=$BATCH --steps=$N --save_freq=$N --log_freq=25 --seed=$TRAIN_SEED \
  --output_dir=/tmp/cl/train_$NAME > /kaggle/working/cl/train_$NAME.log 2>&1
RC=$?
kill $MON
echo "[gpu$GPU] train $NAME: exit=$RC steps=$N minutes=$(( ($(date +%s) - S) / 60 )) peakMiB=$(sort -n /tmp/mem_$NAME.txt | tail -1)" >> /kaggle/working/cl/progress.log
[ $RC -ne 0 ] && tail -30 /kaggle/working/cl/train_$NAME.log >> /kaggle/working/cl/progress.log
exit $RC
EOF
chmod +x /tmp/train_one.sh

# Smoke: a few CIVD steps; checks the launcher, teacher loading, memory, and distill ~0 at step 1
/tmp/train_one.sh 1 smoke $SMOKE_STEPS civd
RC=$?
grep "\[civd\]" /kaggle/working/cl/train_smoke.log | tee -a /kaggle/working/cl_report.txt
tail -2 /kaggle/working/cl/progress.log | tee -a /kaggle/working/cl_report.txt
rm -rf /tmp/cl/train_smoke
[ $RC -eq 0 ] || { echo "CIVD smoke failed; M0 only" | tee -a /kaggle/working/cl_report.txt; touch /tmp/civd_failed; }

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
source /tmp/params.sh
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl
cat > /tmp/eval_one.sh <<'EOF'
#!/usr/bin/env bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
source /tmp/params.sh
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl
GPU=$1; POL=$2; T=$3; NAME=$4
DIR=/kaggle/working/cl/eval/$NAME
[ -f $DIR/eval_info.json ] && exit 0
S=$(date +%s)
MUJOCO_EGL_DEVICE_ID=0 /tmp/venv/bin/python -m lerobot.scripts.lerobot_eval \
  --policy.path=$POL --policy.device=cuda:$GPU --policy.n_action_steps=$NAS '--rename_map={"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}' \
  --env.type=libero --env.task=$SUITE "--env.task_ids=[$T]" --env.max_parallel_tasks=1 \
  --eval.n_episodes=$EVAL_EPISODES --eval.batch_size=5 --seed=1000 \
  --output_dir=$DIR > $DIR.log 2>&1
RC=$?
echo "[gpu$GPU] eval $NAME: exit=$RC seconds=$(( $(date +%s) - S )) $(grep -o '"pc_success": [0-9.]*' $DIR/eval_info.json 2>/dev/null | tail -1)" >> /kaggle/working/cl/progress.log
[ $RC -ne 0 ] && tail -15 $DIR.log >> /kaggle/working/cl/progress.log
exit 0
EOF
chmod +x /tmp/eval_one.sh

# Each GPU: train its method, then evaluate it on the new task first, then the old tasks
run() {
  GPU=$1; NAME=$2; MODE=$3; LAM=$4
  /tmp/train_one.sh $GPU $NAME $STEPS $MODE $LAM || return
  CK=$(ls -d /tmp/cl/train_$NAME/checkpoints/*/pretrained_model | sort | tail -1)
  for T in $NEW_TASK $OLD_TASKS; do /tmp/eval_one.sh $GPU $CK $T ${NAME}_task$T; done
}
START=$(date +%s)
for G in 0 1; do
  V=RUN$G; read NAME MODE LAM <<< "${!V}"
  [ -z "$NAME" ] && continue
  if [ "$MODE" != "seqft" ] && [ -f /tmp/civd_failed ]; then continue; fi
  run $G $NAME $MODE $LAM &
done
wait
echo "=== CL done in $(( ($(date +%s) - START) / 60 )) min" | tee -a /kaggle/working/cl_report.txt
for f in /kaggle/working/cl/train_*.log; do grep "\[civd\]" $f | tail -2 | sed "s|^|$(basename $f): |"; done | tee -a /kaggle/working/cl_report.txt
cat /kaggle/working/cl/progress.log | tee -a /kaggle/working/cl_report.txt

In [ ]:
import json, math, re
from pathlib import Path

out = Path("/kaggle/working/cl")
p = {k: v.strip('"') for k, v in (l.strip().split("=", 1) for l in open("/tmp/params.sh") if "=" in l)}
new, old = int(p["NEW_TASK"]), [int(t) for t in p["OLD_TASKS"].split()]
runs = [p[k].split() for k in ("RUN0", "RUN1") if p.get(k)]
names = [r[0] for r in runs]
labels = {r[0]: f"{r[0]} {r[1]}" + (f" λ={r[2]}" if "civd" in r[1] else "") for r in runs}
# Stage 0 = R1b (lerobot/smolvla_libero, n_action_steps 10, seed 1000, 10 episodes), measured in 02_r2_calibration
R1B = {0: 10, 1: 10, 2: 8, 3: 6, 4: 9, 5: 10, 6: 7, 7: 9, 8: 10, 9: 9}

def wilson(k, n, z=1.96):
    q = k / n; d = 1 + z * z / n; c = (q + z * z / (2 * n)) / d
    h = z * math.sqrt(q * (1 - q) / n + z * z / (4 * n * n)) / d
    return f"{100 * (c - h):.0f}–{100 * (c + h):.0f}"

def sr(name):
    f = out / "eval" / name / "eval_info.json"
    if not f.exists():
        return None
    o = json.load(open(f))["overall"]
    return int(o["n_success"]), int(o["n_episodes"])

lines = [f"# Phase 4 · one CL stage on LIBERO-Goal · base {p['BASE']} · new task {new} · {p['STEPS']} steps × batch {p['BATCH']} · train seed {p['TRAIN_SEED']}", "",
         "| task | role | stage 0 (R1b) | " + " | ".join(labels[n] for n in names) + " |", "|---|---|---|" + "---|" * len(names)]
summary = {}
for t in [new] + old:
    cells = []
    for m in names:
        r = sr(f"{m}_task{t}")
        cells.append("missing" if r is None else f"{r[0]}/{r[1]} ({wilson(*r)})")
        summary[(m, t)] = r
    lines.append(f"| {t} | {'new' if t == new else 'old'} | {R1B[t]}/10 | " + " | ".join(cells) + " |")
lines += ["", "| method | new-task gain (pp) | mean forgetting on old tasks (pp) |", "|---|---|---|"]
for m, label in [(n, labels[n]) for n in names]:
    rn = summary.get((m, new))
    ro = [summary.get((m, t)) for t in old]
    gain = f"{10 * (rn[0] * 10 / rn[1] - R1B[new]):+.0f}" if rn else "n/a"
    fg = [10 * (R1B[t] - r[0] * 10 / r[1]) for t, r in zip(old, ro) if r]
    lines.append(f"| {label} | {gain} | {sum(fg) / len(fg):.0f} ({len(fg)} tasks) |" if fg else f"| {label} | {gain} | n/a |")
for m in names:
    log = (out / f"train_{m}.log").read_text(errors="ignore") if (out / f"train_{m}.log").exists() else ""
    st = re.findall(r"step:(\S+).*?loss:(\S+).*?updt_s:(\S+).*?mem_gb:(\S+)", log)
    if st:
        u = [float(s[2]) for s in st]
        lines.append(f"- {m}: last loss {st[-1][1]} at step {st[-1][0]}; update {sum(u) / len(u):.2f} s/step; mem {st[-1][3]} GB")
(out / "summary.md").write_text("\n".join(lines))
print("\n".join(lines))